In [10]:
#!pip install seaborn

In [1]:
import os 
from tqdm import tqdm
import pandas as pd
import subprocess
import pickle
import ast
import numpy as np

import matplotlib.pyplot as plt

In [2]:
red_db_file  = "../2_compiling_unified_database_files/our_database_12_03_2025_red.csv"
red_db = pd.read_csv(red_db_file)
red_db.head()

,File name,Color,Polymer,Matching,4000.0,3999.0,3998.0,3997.0,3996.0,3995.0,...,459.0,458.0,457.0,456.0,455.0,454.0,453.0,452.0,451.0,450.0
0,Adv1.1_3.csv,NaN,PE,0.98,0.001491,0.001483,0.001483,0.001492,0.001509,0.001532,...,0.023790,0.027825,0.032067,0.032876,0.030447,0.027955,0.027342,0.028074,0.028446,0.027935
1,Adv1.1_4.csv,white,PE,0.95,0.002901,0.002914,0.002915,0.002900,0.002873,0.002843,...,0.056986,0.056147,0.054138,0.051779,0.050458,0.050193,0.049825,0.048727,0.047823,0.047667
2,Adv1.1_5.csv,transparent,PE,0.92,0.002795,0.002797,0.002800,0.002804,0.002808,0.002809,...,0.097227,0.100712,0.101276,0.097796,0.091534,0.085392,0.081877,0.082037,0.083644,0.084670
3,Adv1.1_6.csv,blue,PP,0.96,0.001478,0.001477,0.001489,0.001509,0.001533,0.001553,...,0.018965,0.019365,0.019543,0.018910,0.018044,0.017325,0.016312,0.014865,0.013980,0.013988
4,Adv1.1_7.csv,black,PE,0.98,0.008695,0.008685,0.008666,0.008636,0.008603,0.008585,...,0.043483,0.044460,0.044129,0.043891,0.045927,0.049857,0.052803,0.052295,0.049565,0.047077


In [3]:
filenames = list(red_db['File name'])

In [13]:
#CNN1D

results = {}
for file_name in tqdm(filenames):
    # Define the command to be executed
    command = f"python -m src.infer -i ../1_converting_sp_to_csv/csv_files/{file_name} -m cnn1d_v0.2.0.onnx --model-name cnn -k 1"

    # Run the command and capture the output
    output = subprocess.check_output(command, shell=True).decode("utf-8")

    # Process the output to convert it into a list
    output_list = output.splitlines()

    results.update({file_name : output_list})

100%|███████████████████████████████████████| 2010/2010 [42:49<00:00,  1.28s/it]


In [14]:
results_red = {}
for i in tqdm(results.keys()):
    str_representation = eval(results[i][0])
    result_list = str_representation    
    results_red.update({i : result_list})

with open('cnn1d_results_full_db.pickle', 'wb') as file:
    pickle.dump(results_red, file)

100%|████████████████████████████████████| 2010/2010 [00:00<00:00, 11623.84it/s]


In [8]:
#import pickle
#
#with open('cnn1d_results_full_db.pickle', 'rb') as file:
#    results_red = pickle.load(file)

In [16]:
# Transform the dictionary into a DataFrame
rows = []
for file_name, values in results_red.items():
    if len(values[0][0]) == 2 and values[0][0] == ['HDPE', 'LDPE']:
        rows.append({'File name': file_name, 'Polymer': 'PE', 'Matching': values[0][1], 'if_mixture': 'no'})
    elif len(values[0][0]) == 2:

        if values[0][0][0] == 'LDPE':
            pol1 = 'PE'
            pol2 = values[0][0][1]
        elif values[0][0][1] == 'LDPE':
            pol1 = 'PE'
            pol2 = values[0][0][0]
            
        elif values[0][0][0] == 'HDPE':
            pol1 = 'PE'
            pol2 = values[0][0][1]
        elif values[0][0][1] == 'HDPE':
            pol1 = 'PE'
            pol2 = values[0][0][0]
            
        elif values[0][0][0] == 'PP':
            pol1 = 'PP'
            pol2 = values[0][0][1]
        elif values[0][0][1] == 'PP':
            pol1 = 'PP'
            pol2 = values[0][0][0]
            
        elif values[0][0][0] == 'PS':
            pol1 = 'PS'
            pol2 = values[0][0][1]
        elif values[0][0][1] == 'PS':
            pol1 = 'PS'
            pol2 = values[0][0][0]
            
        elif values[0][0][0] == 'CPE':
            pol1 = 'CPE'
            pol2 = values[0][0][1]
        elif values[0][0][1] == 'CPE':
            pol1 = 'CPE'
            pol2 = values[0][0][0]
            
        else:
            pol1 = values[0][0][0]
            pol2 = values[0][0][1]
            
        rows.append({'File name': file_name, 'Polymer': pol1+'+'+pol2, 'Matching': values[0][1], 'if_mixture': 'yes'})
    elif len(values[0][0]) == 1 and values[0][0][0] == 'HDPE':
        rows.append({'File name': file_name, 'Polymer': 'PE', 'Matching': values[0][1], 'if_mixture': 'no'})
    elif len(values[0][0]) == 1 and values[0][0][0] == 'LDPE':
        rows.append({'File name': file_name, 'Polymer': 'PE', 'Matching': values[0][1], 'if_mixture': 'no'})
    elif len(values[0][0]) == 1:
        rows.append({'File name': file_name, 'Polymer': values[0][0][0], 'Matching': values[0][1], 'if_mixture': 'no'})

# Create a DataFrame
results_red_first = pd.DataFrame(rows)
results_red_first = results_red_first.merge(red_db[['File name']], on='File name', how='inner')

data = {'File name': results_red_first['File name'].to_list(), 'true' : red_db['Polymer'].to_list(), 'predicted' : results_red_first['Polymer'].to_list()}
df = pd.DataFrame(data)
df

,File name,true,predicted
0,Adv1.1_3.csv,PE,PE
1,Adv1.1_4.csv,PE,PE
2,Adv1.1_5.csv,PE,PE+PA
3,Adv1.1_6.csv,PP,PP
4,Adv1.1_7.csv,PE,PE
...,...,...,...
2005,Templfj_3_35.csv,PP,PP+PVC
2006,Templfj_3_36.csv,PE,PE
2007,Templfj_3_37.csv,PS,PS
2008,Templfj_3_38.csv,PS,PS


In [17]:
df.to_csv('manual_and_CNN1D_classification.csv', index=False)